In [ ]:
from google.colab import files
files.upload()

{}

In [ ]:
!pip install torch scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    BaggingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay,
)


# Cònig
TRAIN_FILE   = "train_val_dataset_1e-8.txt"  # file train
TEST_FILE    = "test_dataset_1e-8.txt"        # file test
OUTPUT_ROOT  = "AutoEncoder"                  # output root file
THRESHOLD    = 0.5                            # threshold

LATENT_DIM   = 64    # latent space
# total_loss = ALPHA * loss_recon + BETA * loss_class
ALPHA        = 0.8   # MSELoss
BETA         = 0.2   # BCEWithLogitsLoss

# Hyperparameter
BATCH_SIZE   = 64
MAX_EPOCHS   = 200
LR_AE        = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 50

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device  : {DEVICE}")
print(f"[INFO] Output  : {OUTPUT_ROOT}/")
print(f"[INFO] α={ALPHA} (recon)  β={BETA} (class)  "
      f"→ total_loss = {ALPHA}·MSE + {BETA}·BCEWithLogits")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 – DATA PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

train_df = pd.read_csv(TRAIN_FILE, sep=" ")
test_df  = pd.read_csv(TEST_FILE,  sep=" ")
print(f"  Train shape : {train_df.shape}")
print(f"  Test  shape : {test_df.shape}")

X_train_raw = train_df.drop(columns=["RID", "Phenotype"]).values.astype(np.float32)
y_train     = train_df["Phenotype"].values.astype(np.float32)
X_test_raw  = test_df.drop(columns=["RID",  "Phenotype"]).values.astype(np.float32)
y_test      = test_df["Phenotype"].values.astype(int)

INPUT_DIM = X_train_raw.shape[1]
print(f"  SNP features : {INPUT_DIM}")
print(f"  Train labels – Control:{(y_train==0).sum():.0f}  "
      f"Alzheimer:{(y_train==1).sum():.0f}")
print(f"  Test  labels – Control:{(y_test ==0).sum()}  "
      f"Alzheimer:{(y_test ==1).sum()}")

scaler     = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_sc  = scaler.transform(X_test_raw).astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 – SUPERVISED AUTOENCODER
# ══════════════════════════════════════════════════════════════════════════════

class SupervisedAutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 64):
        super().__init__()

        #  Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor):
        latent        = self.encoder(x)
        reconstructed = self.decoder(latent)
        logits        = self.classifier(latent)
        return reconstructed, latent, logits


# DataLoader
X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_t = torch.tensor(y_train,    dtype=torch.float32)
full_ds   = TensorDataset(X_train_t, y_train_t)

n_val      = max(1, int(0.1 * len(full_ds)))
n_train_ae = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train_ae, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"  DataLoader – Train: {len(train_ds)} sample | Val: {len(val_ds)} sample")

# Model Initialization
model = SupervisedAutoEncoder(INPUT_DIM, LATENT_DIM).to(DEVICE)

criterion_recon = nn.MSELoss()            # reconstruction loss
criterion_class = nn.BCEWithLogitsLoss()  # classification loss

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR_AE, weight_decay=WEIGHT_DECAY
)

n_params = sum(p.numel() for p in model.parameters())
print(f"  Tổng parameters      : {n_params:,}")
print(f"  Loss formula         : {ALPHA}·BCELoss + {BETA}·BCEWithLogitsLoss")
print(f"  Optimizer            : AdamW (lr={LR_AE}, wd={WEIGHT_DECAY})")
print(f"  Epochs={MAX_EPOCHS} | Batch={BATCH_SIZE} | Patience={PATIENCE}\n")

best_val_total   = float("inf")
patience_counter = 0
best_state_dict  = None

# Lưu lịch sử cả 3 loại loss để vẽ biểu đồ
history = {
    "train_recon": [], "train_class": [], "train_total": [],
    "val_recon":   [], "val_class":   [], "val_total":   [],
}

for epoch in range(1, MAX_EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    t_recon = t_class = t_total = 0.0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE).unsqueeze(1)   # [B] → [B, 1] cho BCEWithLogitsLoss

        optimizer.zero_grad()
        reconstructed, latent, logits = model(xb)

        loss_recon = criterion_recon(reconstructed, xb)
        loss_class = criterion_class(logits, yb)
        total_loss = ALPHA * loss_recon + BETA * loss_class  # gộp loss có trọng số

        total_loss.backward()
        optimizer.step()

        n = xb.size(0)
        t_recon += loss_recon.item() * n
        t_class += loss_class.item() * n
        t_total += total_loss.item() * n

    N_tr   = len(train_ds)
    t_recon /= N_tr; t_class /= N_tr; t_total /= N_tr

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    v_recon = v_class = v_total = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).unsqueeze(1)

            reconstructed, latent, logits = model(xb)

            loss_recon = criterion_recon(reconstructed, xb)
            loss_class = criterion_class(logits, yb)
            total_loss = ALPHA * loss_recon + BETA * loss_class

            n = xb.size(0)
            v_recon += loss_recon.item() * n
            v_class += loss_class.item() * n
            v_total += total_loss.item() * n

    N_val  = len(val_ds)
    v_recon /= N_val; v_class /= N_val; v_total /= N_val
    # History
    history["train_recon"].append(t_recon)
    history["train_class"].append(t_class)
    history["train_total"].append(t_total)
    history["val_recon"].append(v_recon)
    history["val_class"].append(v_class)
    history["val_total"].append(v_total)

    # Log
    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{MAX_EPOCHS}"
              f" │ Train  Recon={t_recon:.5f}  Class={t_class:.5f}  Total={t_total:.5f}"
              f" │ Val    Recon={v_recon:.5f}  Class={v_class:.5f}  Total={v_total:.5f}")

    # Early Stopping
    if v_total < best_val_total - 1e-6:
        best_val_total   = v_total
        patience_counter = 0
        best_state_dict  = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n  ⏹  Early stopping in epoch {epoch}"
                  f"  (best val_total_loss = {best_val_total:.5f})")
            break

model.load_state_dict(best_state_dict)
print(f"\n  Supervised AutoEncoder training complete  ✓")
print(f"  Best val_total_loss = {best_val_total:.5f}")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
ep_axis = range(1, len(history["train_total"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Supervised AutoEncoder – Loss Curves", fontsize=13)

loss_pairs = [
    ("Reconstruction Loss (MSE)",   "train_recon", "val_recon",  "#2196F3"),
    ("Classification Loss (BCE)",   "train_class", "val_class",  "#4CAF50"),
    ("Total Loss (α·MSE + β·BCE)",  "train_total", "val_total",  "#FF9800"),
]
for ax, (title, tr_key, vl_key, color) in zip(axes, loss_pairs):
    ax.plot(ep_axis, history[tr_key], label="Train", color=color, lw=2)
    ax.plot(ep_axis, history[vl_key], label="Val",   color=color, lw=2,
            linestyle="--", alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.tight_layout()
loss_path = os.path.join(OUTPUT_ROOT, "supervised_ae_loss_curves.pdf")
fig.savefig(loss_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {loss_path}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 – FEATURE EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

for param in model.parameters():
    param.requires_grad = False
model.eval()


def extract_features(X_np: np.ndarray) -> np.ndarray:
    tensor = torch.tensor(X_np, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        _, latent, _ = model(tensor)
    return latent.cpu().numpy()


Z_train = extract_features(X_train_sc)
Z_test  = extract_features(X_test_sc)

print(f"  Encoded train : {Z_train.shape}")
print(f"  Encoded test  : {Z_test.shape}")
print(f"  Latent mean (train) : {Z_train.mean():.4f}  std: {Z_train.std():.4f}")
print(f"  Latent mean (test)  : {Z_test.mean():.4f}  std: {Z_test.std():.4f}")
print("  Feature extraction complete  ✓")

y_train_int = y_train.astype(int)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 – CLASSIFICATION (Scikit-learn)
# ══════════════════════════════════════════════════════════════════════════════


classifiers = {
    "LR"              : LogisticRegression(class_weight="balanced", max_iter=1000,
                                           random_state=SEED),
    "SVM_RBF"         : SVC(kernel="rbf",    class_weight="balanced", probability=True,
                            random_state=SEED),
    "SVM_Linear"      : SVC(kernel="linear", class_weight="balanced", probability=True,
                            random_state=SEED),
    "RandomForest"    : RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                               max_features="sqrt", random_state=SEED,
                                               n_jobs=-1),
}

for name in classifiers:
    classifiers[name].fit(Z_train, y_train_int)
    print(f"  {name} trained  ✓")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 – EVALUATION & VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_and_plot(name: str,
                      y_true: np.ndarray,
                      y_probs: np.ndarray,
                      threshold: float,
                      model_dir: str):

    os.makedirs(model_dir, exist_ok=True)
    y_preds = (y_probs >= threshold).astype(int)

    metrics = {
        "Accuracy" : accuracy_score(y_true, y_preds),
        "Precision": precision_score(y_true, y_preds, zero_division=0),
        "Recall"   : recall_score(y_true, y_preds,    zero_division=0),
        "F1-Score" : f1_score(y_true, y_preds,         zero_division=0),
        "AUC-ROC"  : roc_auc_score(y_true, y_probs),
    }

    # ── 1. Confusion Matrix ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_preds,
        display_labels=["Control", "Alzheimer"],
        cmap="Blues", values_format="d", ax=ax,
    )
    ax.set_title(f"Confusion Matrix – {name}", fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "confusion_matrix.pdf"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ── 2. ROC Curve ──────────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="darkorange", lw=2,
            label=f"ROC curve (AUC = {metrics['AUC-ROC']:.4f})")
    ax.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve – {name}", fontsize=12)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "roc_curve.pdf"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ── 3. Risk Score Distribution (KDE) ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.kdeplot(y_probs[y_true == 0], label="Control",
                fill=True, color="green", alpha=0.5, ax=ax)
    sns.kdeplot(y_probs[y_true == 1], label="Alzheimer",
                fill=True, color="red", alpha=0.5, ax=ax)
    ax.axvline(x=threshold, color="black", linestyle="--", lw=1.5,
               label=f"Threshold = {threshold}")
    ax.set_xlabel("Predicted Probability")
    ax.set_ylabel("Density")
    ax.set_title(f"Risk Score Distribution – {name}", fontsize=12)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "risk_score_distribution.pdf"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    return metrics, y_preds


# ── Vòng lặp đánh giá tất cả mô hình ─────────────────────────────────────────
all_metrics    = []
predictions_df = pd.DataFrame({"y_true": y_test})

print(f"\n  {'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}")
print("  " + "─"*58)

for name, clf in classifiers.items():
    y_prob    = clf.predict_proba(Z_test)[:, 1]
    model_dir = os.path.join(OUTPUT_ROOT, name)

    metrics, y_preds = evaluate_and_plot(
        name, y_test, y_prob, THRESHOLD, model_dir
    )

    print(f"  {name:<22} {metrics['Accuracy']:>7.4f} {metrics['Precision']:>7.4f}"
          f" {metrics['Recall']:>7.4f} {metrics['F1-Score']:>7.4f}"
          f" {metrics['AUC-ROC']:>7.4f}")

    all_metrics.append({
        "Model"    : name,
        "Threshold": THRESHOLD,
        **{k: round(v, 4) for k, v in metrics.items()},
    })

    predictions_df[f"{name}_prob"] = y_prob
    predictions_df[f"{name}_pred"] = y_preds

print("  " + "─"*58)

# ── Lưu CSV ───────────────────────────────────────────────────────────────────
metrics_df = pd.DataFrame(all_metrics).sort_values("AUC-ROC", ascending=False)
metrics_df.to_csv(os.path.join(OUTPUT_ROOT, "metrics_summary.csv"),    index=False)
predictions_df.to_csv(os.path.join(OUTPUT_ROOT, "test_predictions.csv"), index=False)

print(f"\n  Saved: {OUTPUT_ROOT}/metrics_summary.csv")
print(f"  Saved: {OUTPUT_ROOT}/test_predictions.csv")

# ── In bảng kết quả cuối (sorted by AUC-ROC) ─────────────────────────────────
print("\n" + "="*72)
print("  FINAL RESULTS (sorted by AUC-ROC)")
print("="*72)
print(f"  {'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}")
print("  " + "─"*62)
for _, row in metrics_df.iterrows():
    star = "  ◀ BEST" if row["Model"] == metrics_df.iloc[0]["Model"] else ""
    print(f"  {row['Model']:<22} {row['Accuracy']:>7.4f} {row['Precision']:>7.4f}"
          f" {row['Recall']:>7.4f} {row['F1-Score']:>7.4f}"
          f" {row['AUC-ROC']:>7.4f}{star}")
print("="*72)

# ── Tổng kết cấu trúc output ──────────────────────────────────────────────────
all_pdfs = sorted(glob.glob(os.path.join(OUTPUT_ROOT, "**", "*.pdf"), recursive=True))
print(f"\n  ✓ Tổng cộng {len(all_pdfs)} PDF files")
print(f"  ✓ 2 CSV files (metrics_summary, test_predictions)")
print(f"\n  Cấu trúc thư mục output:")
print(f"  {OUTPUT_ROOT}/")
print(f"  ├── supervised_ae_loss_curves.pdf   ← Recon / Class / Total loss")
print(f"  ├── metrics_summary.csv")
print(f"  ├── test_predictions.csv")
for clf_name in classifiers:
    print(f"  ├── {clf_name}/")
    print(f"  │   ├── confusion_matrix.pdf")
    print(f"  │   ├── roc_curve.pdf")
    print(f"  │   └── risk_score_distribution.pdf")

print("\n" + "="*60)
print("  PIPELINE COMPLETE  ✓")
print("="*60)

[INFO] Device  : cuda
[INFO] Output  : AutoEncoder/
[INFO] α=0.8 (recon)  β=0.2 (class)  → total_loss = 0.8·MSE + 0.2·BCEWithLogits

STEP 1 – DATA PREPROCESSING
  Train shape : (1000, 1726)
  Test  shape : (174, 1726)
  SNP features : 1724
  Train labels – Control:486  Alzheimer:514
  Test  labels – Control:84  Alzheimer:90
  MinMaxScaler fit trên TRAIN, transform trên TEST  ✓
  (Tập Test KHÔNG được đưa vào DataLoader của AE)

STEP 2 – SUPERVISED AUTOENCODER (PyTorch)
  DataLoader – Train: 900 mẫu | Val: 100 mẫu
  (Tập Test KHÔNG được đưa vào DataLoader)

  Kiến trúc Encoder    : 1724 → 256 → 128 → 64 (linear)
  Kiến trúc Decoder    : 64 → 128 → 256 → 1724 (sigmoid)
  Kiến trúc Classifier : 64 → 32 → 1 (logit, no sigmoid)
  Tổng parameters      : 969,277
  Loss formula         : 0.8·BCELoss + 0.2·BCEWithLogitsLoss
  Optimizer            : AdamW (lr=0.001, wd=0.0001)
  Epochs=200 | Batch=64 | Patience=50

  Epoch    1/200 │ Train  Recon=0.13692  Class=0.69731  Total=0.24900 │ Val    Rec

In [ ]:
import shutil
from google.colab import files

# Create a zip archive of the AutoEncoder directory
output_filename = 'SAE_results_1e-8'
shutil.make_archive(output_filename, 'zip', OUTPUT_ROOT)

print(f"Directory '{OUTPUT_ROOT}' compressed to '{output_filename}.zip'")

# Download the zip file
files.download(f'{output_filename}.zip')

Directory 'AutoEncoder' compressed to 'SAE_results_1e-8_MSE.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>